# Notebook 8: Memory in Chains and Agents

**🎯 Goal:** Learn how to add memory to both simple conversational chains and complex, tool-using agents to build stateful applications that remember past interactions.

## 🧩 Concept Overview

Memory is what separates a one-off Q&A bot from a true conversational assistant. It allows your AI to remember what was said previously, providing context for new interactions. This is crucial for both simple chatbots and for advanced agents that need to remember the results of their actions.

We will cover:
1.  **Memory in Chains:** Using `ConversationChain` to easily build chatbots.
2.  **Memory in Agents:** Giving agents a memory of past user interactions and tool uses.

## 🖼️ Visual Diagram

Memory is a component that feeds context into the prompt at each step.

```
                                +--------------------+
                                |   Memory Buffer    | (e.g., chat history)
                                +---------+----------+
                                          ^      |
                                (Save)    |      | (Injects History)
                                          |      v
User Query ---> [Chain or Agent] ---> AI Output ---+---> Final Response
                       ^                               
                       |                               
         (Prompt is now aware of past conversation)    
```

## ⚙️ Setup

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory, ConversationBufferWindowMemory
from langchain.agents import Tool, AgentExecutor, create_react_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain import hub

load_dotenv()
llm = ChatOpenAI(model="gpt-4o", temperature=0.5)

## 1. Memory in Chains

This is the most straightforward way to build a chatbot. `ConversationChain` combined with a memory object handles everything for you.

In [ ]:
# We'll use ConversationBufferWindowMemory, which remembers the last 'k' interactions.
# It's a good balance of context and efficiency.
memory_for_chain = ConversationBufferWindowMemory(k=3)

conversation_chain = ConversationChain(
    llm=llm,
    memory=memory_for_chain,
    verbose=False # Set to True to see the full prompt
)

# Turn 1
response1 = conversation_chain.invoke({"input": "Hi! I'm Bob."})
print(f"AI: {response1['response']}")

# Turn 2
response2 = conversation_chain.invoke({"input": "I'm interested in learning about artificial intelligence."})
print(f"AI: {response2['response']}")

# Turn 3
response3 = conversation_chain.invoke({"input": "Do you remember my name and what I'm interested in?"})
print(f"AI: {response3['response']}")

## 2. Memory in Agents

Adding memory to an agent is slightly more involved but incredibly powerful. It allows the agent to remember not just what the user said, but also what tools it used and what the results were.

To do this, we need two things:
1.  A `memory` object, just like before.
2.  A special `MessagesPlaceholder` in our prompt to tell the agent where to insert the history.

In [ ]:
# The prompt now needs a placeholder for the chat history
agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="chat_history"), # Where the memory will be injected
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# Create a simple search tool
def search_function(query: str):
    """A mock search tool."""
    return f"Results for '{query}': LangGraph is a new library for building stateful agents."

tools = [Tool(name="Search", func=search_function, description="Searches for information.")]

# We'll use ConversationBufferMemory to store the history
memory_for_agent = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# Create the agent
agent = create_react_agent(llm, tools, agent_prompt)

# Create the agent executor, now with memory!
agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    memory=memory_for_agent, 
    verbose=True
)

### Chatting with the Agent

Let's see if it can remember the context from a tool call.

In [ ]:
# First, ask it to perform a search
agent_executor.invoke({"input": "Can you search for info on LangGraph?"})

# Now, ask a follow-up question that relies on the previous context
agent_executor.invoke({"input": "What is that library used for?"})

## 🔧 Hands-On Exercise

**Goal:** Build a conversational agent that remembers a user's preference.

1.  Create an agent with a tool called `save_preference` that takes a `preference` string and returns a confirmation message (e.g., "Preference saved!").
2.  Add `ConversationBufferMemory` to the agent executor.
3.  In the first turn, tell the agent: `"My favorite topic is space exploration."` The agent should use the tool to save this.
4.  In the second turn, ask: `"What's my favorite topic?"` The agent should be able to answer from its memory.

In [ ]:
# 1. Create the tool
@tool
def save_preference(preference: str) -> str:
    """Saves a user's preference to a (mock) database."""
    print(f"-- Saving preference: {preference} --")
    return "Preference saved successfully!"

exercise_tools = [save_preference]

# 2. Create the agent with memory
exercise_memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
exercise_agent = create_react_agent(llm, exercise_tools, agent_prompt)
exercise_executor = AgentExecutor(agent=exercise_agent, tools=exercise_tools, memory=exercise_memory, verbose=True)

# 3. First turn
exercise_executor.invoke({"input": "Please remember that my favorite topic is space exploration."})

# 4. Second turn
exercise_executor.invoke({"input": "Do you know what my favorite topic is?"})

## 🐞 Bug Bounty

The code below correctly creates an agent and a memory object, but it forgets to pass the `memory` to the `AgentExecutor`.

**Task:** Fix the `AgentExecutor` initialization by adding the `memory` object.

In [ ]:
# --- BROKEN CODE ---
buggy_memory = ConversationBufferMemory(memory_key="chat_history")

# The executor is created without the memory object!
buggy_executor = AgentExecutor(agent=agent, tools=tools, verbose=False) 

buggy_executor.invoke({"input": "My name is Alice."})
response = buggy_executor.invoke({"input": "What's my name?"})
print(f"Buggy response: {response['output']}") # It won't know the name

# --- FIXED CODE ---
fixed_memory = ConversationBufferMemory(memory_key="chat_history")
fixed_executor = AgentExecutor(agent=agent, tools=tools, memory=fixed_memory, verbose=False)

fixed_executor.invoke({"input": "My name is Alice."})
response = fixed_executor.invoke({"input": "What's my name?"})
print(f"Fixed response: {response['output']}")

## 💡 Pro Tip

When adding memory to an agent, you **must** include a `MessagesPlaceholder` in your prompt. The `memory_key` in your memory object (e.g., `memory_key="chat_history"`) must match the `variable_name` in the placeholder (e.g., `MessagesPlaceholder(variable_name="chat_history")`). This is how the `AgentExecutor` knows where to inject the conversation history into the prompt.

## 🏁 Summary

You can now build AI that remembers!

1.  **Memory is for Conversations:** It provides the context of past interactions, which is essential for building chatbots and stateful assistants.
2.  **`ConversationChain` is for simple chats:** It's the easiest way to get a conversational bot up and running.
3.  **Agents Need Memory for Complex Tasks:** Adding memory to an `AgentExecutor` allows an agent to remember the results of its past actions and have multi-turn conversations with the user.